# Lecture 8 — Class Exercise
## Choropleth Maps

> **Push to:** `week08/lecture08_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:
# Task 1
gm = px.data.gapminder()
gm_2007 = gm[gm['year'] == 2007].copy()

global_mean = gm_2007['lifeExp'].mean()
gm_2007['deviation'] = gm_2007['lifeExp'] - global_mean

fig = px.choropleth(
    gm_2007,
    locations='iso_alpha',
    color='deviation',
    hover_name='country',
    hover_data={'lifeExp': ':.1f', 'deviation': ':.1f', 'iso_alpha': False},
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    labels={
        'deviation': 'Δ from global mean (yrs)',
        'lifeExp': 'Life Expectancy (yrs)',
    },
    title='Sub-Saharan Africa Falls 20+ Years Below World Average Life Expectancy (2007)',
)
fig.show()

## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** Built-in Plotly `px.data.election_geojson()` — Montreal electoral districts (2013 mayoral election)

**Chart type chosen** (`px.choropleth_map`) **and reason:**

`px.choropleth_map` because the data covers city-level geography where a tile-backed map with zoom and pan provides essential street and neighbourhood context. A flat projection (`px.choropleth`) would offer no spatial reference at this scale.

**Colour scale chosen** (sequential) **and reason:**

Sequential (`Blues`) because Bergeron's vote share has a natural floor at 0 % and rises in one direction — there is no meaningful "negative" share, so a diverging scale would misrepresent the data.

In [ ]:
# Task 2 — Montreal 2013 Mayoral Election: Bergeron Vote Share by District

geojson = px.data.election_geojson()
df = px.data.election()

# Inspect properties to confirm featureidkey
print(geojson['features'][0]['properties'])

df['bergeron_pct'] = (df['Bergeron'] / df['total'] * 100).round(1)

fig = px.choropleth_map(
    df,
    geojson=geojson,
    locations='district',
    featureidkey='properties.district',
    color='bergeron_pct',
    color_continuous_scale='Blues',
    hover_name='district',
    hover_data={'bergeron_pct': ':.1f', 'Bergeron': True, 'total': True},
    labels={'bergeron_pct': 'Bergeron vote share (%)'},
    map_style='carto-positron',
    zoom=9,
    center={'lat': 45.51, 'lon': -73.61},
    title="Bergeron Sweeps Eastern Montréal with 50%+ Vote Share in 2013 Mayoral Race",
)
fig.show()